# CPRA walkthrough — diverse solutions in a single GNN run

This notebook is a minimal tour of **CPRA** (*Continual Parallel Relaxation Annealing*), the multi-replica extension of CRA-PI-GNN published in *Transactions on Machine Learning Research* (2025): ["Continuous Parallel Relaxation for Finding Diverse Solutions in Combinatorial Optimization Problems"](https://openreview.net/forum?id=ix33zd5zCw).

CPRA trains a **single shared GCN backbone** that produces ``R`` parallel continuous solutions in one forward pass, so one training run yields a *diverse population* of high-quality discrete solutions. QQA4CO ships it under `qqa.pignn.train_cpra_pi_gnn`, sharing the same `GCNNet` backbone and `BinaryRelaxation` penalty as `train_cra_pi_gnn`.

We cover the two diversification regimes separately:

1. **Penalty diversification** — give each replica its own QUBO penalty weight (e.g. one `MaximumIndependentSet` per `penalty` value). One run produces one solution per penalty level, far cheaper than independent runs.
2. **Variation diversification** — every replica solves the *same* problem, but a positive `vari_param` rewards inter-replica spread, so replicas converge to *structurally different* solutions to the same instance.

Both regimes use the same trainer; only the kwargs differ.

> **Hyperparameter note.** Defaults in `train_cpra_pi_gnn` (`init_reg_param=-20`, `annealing_rate=1e-3`, `learning_rate=1e-4`, `num_epochs=1e5`) are tuned for the paper's `N >= 1000` regime. For the small instances in this notebook we use the medium-graph regime `init_reg_param=-2.0`, `annealing_rate=5e-4`, `learning_rate=1e-3`, identical to the CRA-PI-GNN walkthrough.

## 0. Install (run once)

CPRA lives in the optional `pignn` extra (PyTorch Geometric backend):

```bash
pip install "qqa[pignn]"
# or, if you cloned the repo:
uv sync --extra pignn
```

In [ ]:
import time

import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import torch

import qqa
from qqa.pignn import train_cpra_pi_gnn

SEED = 0
qqa.fix_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"qqa version : {qqa.__version__}")
print(f"torch       : {torch.__version__}  (CUDA available: {torch.cuda.is_available()})")
print(f"device      : {DEVICE}")

## 1. A small MIS instance

We use a 60-node Erdős–Rényi graph throughout, small enough that every cell finishes in seconds on CPU but large enough that *different* independent sets exist.

In [ ]:
N = 60
g = nx.erdos_renyi_graph(N, p=0.10, seed=SEED)
print(f"|V| = {g.number_of_nodes()}, |E| = {g.number_of_edges()}")

## 2. Penalty diversification

We sweep four penalty weights ``{1.5, 2.0, 3.0, 5.0}`` for `MaximumIndependentSet`. Each replica owns one penalty value; CPRA returns one solution *per* penalty in a single training run.

> The base `problem` is used only for graph extraction and as a default scoring fallback; per-replica losses come from the corresponding entry in `replica_problems`.

> **Why scale matters.** CPRA shares a single GCN backbone across all replicas, so the four per-replica gradients (each with a different `Q_mat`) compete for the same parameters. On the very small instance here that competition limits how far each replica can travel — the typical use case is `N >= 1000` with `num_epochs >= 1e5` (the `train_cpra_pi_gnn` defaults). What this small-scale demo *does* still show is the real CPRA workflow: rather than launching four independent runs to find the best penalty weight, *one* CPRA run produces four solutions side-by-side, and `result.score['extra']['best_replica']` flags the winner.

In [ ]:
penalty_levels = [1.5, 2.0, 3.0, 5.0]
base_problem = qqa.MaximumIndependentSet(g, penalty=2.0, device=DEVICE)
replica_problems = [qqa.MaximumIndependentSet(g, penalty=p, device=DEVICE) for p in penalty_levels]

t0 = time.time()
result_pen = train_cpra_pi_gnn(
    base_problem,
    num_replicas=len(penalty_levels),
    replica_problems=replica_problems,
    init_reg_param=-2.0,
    annealing_rate=5e-4,
    learning_rate=1e-3,
    num_epochs=4000,
    patience=400,
    check_interval=1000,
    device=DEVICE,
    seed=SEED,
    verbose=False,
)
print(f"runtime: {time.time() - t0:.2f} s")

Inspect the per-replica solutions exposed under `result.score['extra']['replicas']`. For MIS, smaller `loss_fn` corresponds to a larger feasible independent set.

In [ ]:
records = result_pen.score["extra"]["replicas"]
print(f"{'replica':<8} {'penalty':<8} {'|S|':<5} {'feasible':<9} {'obj':<10}")
for rec, pen in zip(records, penalty_levels, strict=True):
    sol = rec["sol"]
    score = rec["score"]
    print(
        f"{rec['replica']:<8} {pen:<8.2f} {int(sol.sum()):<5d} "
        f"{str(score.get('feasible', '?')):<9} {rec['obj']:<10.3f}"
    )
print(f"\nbest replica  : {result_pen.score['extra']['best_replica']}")
print(f"best |S|      : {int(result_pen.best_sol.sum())}")
print(f"best obj      : {result_pen.best_obj:.3f}")

Per-replica training trajectories live in `history['per_replica_obj']` (shape `(epochs, R)`).

In [ ]:
per_repl = np.asarray(result_pen.history["per_replica_obj"])
fig, ax = plt.subplots(1, 1, figsize=(7, 3.5))
for r, pen in enumerate(penalty_levels):
    ax.plot(per_repl[:, r], label=f"penalty={pen}", lw=1.2)
ax.set_xlabel("epoch")
ax.set_ylabel("discrete loss (per replica)")
ax.set_title("CPRA — penalty diversification")
ax.legend(fontsize=8)
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()

## 3. Variation diversification

Same instance for every replica, but `vari_param > 0` adds the diversity term ``-R · Σᵢ stdᵣ(p_{i,r})`` to the loss, which the optimiser minimises by *spreading replicas apart* in the continuous space. The R discrete projections then tend to be structurally different solutions to the *same* problem.

Below we run the same MIS instance twice — once with `vari_param=0.0` (replicas eventually collapse to similar solutions) and once with `vari_param=0.3` — and measure the average pairwise Hamming distance between the four returned solutions.

In [ ]:
def avg_pairwise_hamming(records):
    sols = torch.stack([rec["sol"].float() for rec in records])  # (R, N)
    R = sols.shape[0]
    if R < 2:
        return 0.0
    diffs = []
    for i in range(R):
        for j in range(i + 1, R):
            diffs.append(float((sols[i] != sols[j]).sum().item()))
    return sum(diffs) / len(diffs)


common = dict(
    num_replicas=4,
    init_reg_param=-2.0,
    annealing_rate=5e-4,
    learning_rate=1e-3,
    num_epochs=4000,
    patience=400,
    check_interval=1000,
    device=DEVICE,
    seed=SEED,
    verbose=False,
)

problem_vari = qqa.MaximumIndependentSet(g, penalty=2.0, device=DEVICE)

t0 = time.time()
res_no_vari = train_cpra_pi_gnn(problem_vari, vari_param=0.0, **common)
t_no = time.time() - t0

t0 = time.time()
res_with_vari = train_cpra_pi_gnn(problem_vari, vari_param=0.3, **common)
t_yes = time.time() - t0

h_no = avg_pairwise_hamming(res_no_vari.score["extra"]["replicas"])
h_yes = avg_pairwise_hamming(res_with_vari.score["extra"]["replicas"])

print(
    f"vari_param = 0.0  ->  best |S| = {int(res_no_vari.best_sol.sum())}, "
    f"avg pairwise Hamming = {h_no:.2f}/{N}, runtime = {t_no:.2f}s"
)
print(
    f"vari_param = 0.3  ->  best |S| = {int(res_with_vari.best_sol.sum())}, "
    f"avg pairwise Hamming = {h_yes:.2f}/{N}, runtime = {t_yes:.2f}s"
)

Look at the replica-by-replica solutions for the `vari_param=0.3` run. With diversification on, replicas typically pick *different* vertices for their independent sets — exactly what you want when you need a portfolio of high-quality solutions.

In [ ]:
for rec in res_with_vari.score["extra"]["replicas"]:
    chosen = torch.nonzero(rec["sol"].long(), as_tuple=False).flatten().tolist()
    print(
        f"replica {rec['replica']} | |S|={int(rec['sol'].sum()):2d} | obj={rec['obj']:.3f} | nodes={chosen}"
    )

## 4. CLI usage

Both regimes are also reachable from the `qqa solve` CLI without any Python code:

```bash
# Penalty diversification on a built-in MIS sample (size = 60).
qqa solve --problem mis --size 60 --backend cpra \
  --cpra-num-replicas 4 \
  --cpra-penalty-levels 1.5,2.0,3.0,5.0 \
  --pignn-init-reg-param -2.0 --pignn-annealing-rate 5e-4 \
  --learning-rate 1e-3 --epochs 4000

# Variation diversification (same problem on every replica).
qqa solve --problem mis --size 60 --backend cpra \
  --cpra-num-replicas 4 --cpra-vari-param 0.3 \
  --pignn-init-reg-param -2.0 --pignn-annealing-rate 5e-4 \
  --learning-rate 1e-3 --epochs 4000
```

See `qqa solve --help` for the full flag list.

## 5. Citation

If you use CPRA in published work, please cite:

```bibtex
@article{ichikawa2025cpra,
  title  = {Continuous Parallel Relaxation for Finding Diverse Solutions in Combinatorial Optimization Problems},
  author = {Ichikawa, Yuma and Iwashita, Hiroaki},
  journal= {Transactions on Machine Learning Research},
  year   = {2025},
  url    = {https://openreview.net/forum?id=ix33zd5zCw}
}
```

and the underlying CRA-PI-GNN paper:

```bibtex
@inproceedings{ichikawa2024cra,
  title    = {Controlling Continuous Relaxation for Combinatorial Optimization},
  author   = {Ichikawa, Yuma},
  booktitle= {Advances in Neural Information Processing Systems (NeurIPS)},
  year     = {2024}
}
```